# scikit-learn과 예측 Pipeline

## 철강산업 전력사용량 예측 실험

정답이 있는 자료에서 규칙을 배우는 지도학습과 연속형 값을 맞히는 회귀 문제부터 시작합니다. 선형회귀와 Decision Tree가 예측 규칙을 만드는 방식을 작은 예제로 비교한 뒤, 철강산업 자료의 전처리와 모형을 Pipeline으로 구성합니다.


## 이번 교시의 학습목표

수업이 끝나면 다음을 수행할 수 있습니다.

- 지도학습과 회귀 문제를 정의하고 feature `X`, target `y`, 예측값 `ŷ`과 오차 `y-ŷ`를 구분합니다.
- 선형회귀의 기울기·절편과 잔차제곱합의 관계를 설명하고 새로운 입력의 예측값을 계산합니다.
- Decision Tree가 조건으로 자료를 나누고 말단 노드의 target 평균을 회귀 예측값으로 사용하는 원리를 설명합니다.
- train–validation–test의 역할과 `fit–predict–evaluate` 흐름을 설명하고, 세 후보를 같은 validation 행에서 비교한 뒤 선택된 Pipeline 하나를 test에서 최종 평가합니다.
- 숫자형·범주형 열의 전처리를 구분하고 `ColumnTransformer`와 `Pipeline`이 전처리 정보 누수를 막는 실행 순서를 설명합니다.


## 이번 교시에서 사용할 데이터

### 강의 시연: UCI Steel Industry Energy Consumption

| 항목 | 내용 |
|---|---|
| 수집 맥락 | 한국 철강산업의 2018년 전력·운전 기록 |
| 규모 | 35,040행, 15분 간격 1년 자료 |
| 한 행의 의미 | 특정 15분 구간의 전력사용과 운전조건 |
| target | `Usage_kWh` |
| 주요 feature | 무효전력, power factor, `NSM`, `WeekStatus`, `Day_of_week`, `Load_Type` |

`date`는 시간정보이고 `NSM`은 자정 이후 경과 초입니다. `CO2(tCO2)`는 전력사용량과 매우 밀접한 결과 proxy일 수 있으므로 예측시점과 연구목적을 확인한 뒤 제외합니다. 5교시에는 시간순 미래예측이 아니라 같은 기간의 관측을 무작위로 나눈 holdout에서 Pipeline의 구조를 확인합니다.

### 적용 실습: UCI Bike Sharing

2011~2012년의 17,379개 시간별 관측에서 `cnt`를 예측합니다. 날씨·달력·시간 변수를 feature로 사용하고, `cnt`를 구성하는 `casual`과 `registered`는 제외합니다. Steel과 Bike의 분야는 다르지만 `target–feature–split–preprocessing–baseline–metric` 명세는 같은 방식으로 작성합니다.


## 예측문제의 네 요소: X, y, 예측값과 오차

예측 실험은 먼저 한 행에서 네 요소를 구분하는 일로 시작합니다.

| 요소 | 의미 | Steel 자료의 예 |
|---|---|---|
| $X$ | 예측 시점에 사용할 입력열 | 전력·공정 변수, 요일, 부하유형 |
| $y$ | 실제로 예측할 결과열 | `Usage_kWh` |
| $\hat y$ | 모형이 계산한 예측값 | 예측 전력사용량 |
| $e=y-\hat y$ | 같은 행의 실제값과 예측값의 차이 | 양수면 과소예측, 음수면 과대예측 |

숫자형 열이라고 모두 X가 되는 것은 아닙니다. 결과에서 파생된 열과 예측 시점에
알 수 없는 열은 제외하고, 한 행의 X·y·예측값이 같은 관측을 가리키는지 확인합니다.


In [ ]:
import pandas as pd
from IPython.display import display

rows = pd.DataFrame({
    "reactive_power": [18.0, 25.0, 31.0],
    "load_type": ["Light", "Medium", "Maximum"],
    "Usage_kWh": [42.0, 68.0, 91.0],
})
prediction = pd.Series([45.0, 63.0, 96.0], index=rows.index, name="prediction")

view = rows.copy()
view["prediction"] = prediction
view["residual_y_minus_prediction"] = view["Usage_kWh"] - view["prediction"]
view["absolute_error"] = view["residual_y_minus_prediction"].abs()

print("[계산으로 확인] X·y·예측값·오차는 같은 index의 한 관측을 가리켜야 합니다.")
display(view)
print("X 열:", ["reactive_power", "load_type"])
print("y 열: Usage_kWh")
print("잔차가 양수인 행은 실제값보다 작게 예측한 과소예측입니다.")


## 선형회귀: 하나의 직선으로 예측하기

입력 하나인 선형회귀는 절편과 기울기를 이용해 예측값을 계산합니다.

$$\hat y=\beta_0+\beta_1x$$

절편 $\beta_0$는 $x=0$일 때의 예측값이고, 기울기 $\beta_1$은 $x$가 한 단위
달라질 때 예측값이 얼마나 달라지는지를 나타냅니다. 입력이 여러 개이면 각 입력과
계수의 곱을 더합니다. 회귀식이 모든 관계를 직선으로 표현한다는 가정과 예측오차는
별도로 확인해야 합니다.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

x = np.array([1.0, 2.0, 3.0, 4.0])
y = np.array([32.0, 43.0, 51.0, 64.0])
slope, intercept = np.polyfit(x, y, deg=1)
x_new = 5.0
y_new = intercept + slope * x_new

line_table = pd.DataFrame({"x": x, "observed_y": y, "fitted_y": intercept + slope * x})
print("[계산으로 확인] 선형회귀가 저장한 절편과 기울기")
print(f"예측식: y_hat = {intercept:.2f} + {slope:.2f} × x")
print(f"x=5의 예측값: {y_new:.2f}")
display(line_table.round(2))

grid = np.linspace(0.5, 5.2, 100)
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.scatter(x, y, s=55, label="observed")
ax.plot(grid, intercept + slope * grid, color="#2563eb", label="fitted line")
ax.scatter([x_new], [y_new], marker="X", s=90, color="#dc2626", label="x=5 prediction")
ax.set(xlabel="x", ylabel="predicted target", title="One fitted line is applied to every row")
ax.legend()
plt.tight_layout()
plt.show()


## Decision Tree: 조건을 따라 leaf의 평균으로 이동하기

회귀나무는 하나의 직선을 쓰지 않고 입력값에 대한 조건을 차례로 적용합니다.

```text
temperature <= 16?
├─ 예  → 왼쪽 leaf의 학습 target 평균
└─ 아니오 → 오른쪽 leaf의 학습 target 평균
```

새 행은 학습 때 정한 같은 분기조건을 따라 leaf에 도착하고, 그 leaf에 있던 학습
target의 평균을 예측값으로 받습니다. 나무가 지나치게 깊으면 적은 행의 우연한
차이까지 분리할 수 있으므로 깊이는 validation 자료에서 비교합니다.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.tree import DecisionTreeRegressor, export_text

temperature = np.array([10, 12, 14, 18, 20, 22], dtype=float).reshape(-1, 1)
target = np.array([30, 32, 34, 70, 74, 76], dtype=float)
tree = DecisionTreeRegressor(max_depth=1, random_state=42).fit(temperature, target)
prediction = tree.predict(temperature)
leaf = tree.apply(temperature)

branch_table = pd.DataFrame({
    "temperature": temperature.ravel(),
    "observed": target,
    "leaf_id": leaf,
    "leaf_mean_prediction": prediction,
})
print("[계산으로 확인] 깊이 1 tree의 분기와 leaf 평균")
print(export_text(tree, feature_names=["temperature"]))
display(branch_table)

grid = np.linspace(9, 23, 200).reshape(-1, 1)
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.scatter(temperature.ravel(), target, label="observed")
ax.step(grid.ravel(), tree.predict(grid), where="mid", color="#7c3aed", label="tree prediction")
ax.set(xlabel="temperature", ylabel="target", title="A tree predicts the mean stored in each leaf")
ax.legend()
plt.tight_layout()
plt.show()


## 학습자료·검증자료·평가자료의 역할

전체 자료를 한 번에 학습과 평가에 함께 사용하지 않습니다.

```text
전체 자료
├─ train
│  ├─ fit: 전처리 규칙과 후보모형 학습
│  └─ validation: 후보와 설정 비교
└─ test: 선택을 모두 끝낸 뒤 최종 1회 평가
```

validation 결과를 보고 모형이나 설정을 바꿀 수 있지만, test 결과를 본 뒤 다시
바꾸면 test도 선택과정에 참여합니다. 이 수업의 무작위 holdout은 같은 분포의 새
행에 대한 기본 평가이며, 미래기간 예측은 7교시에서 시간순으로 검증합니다.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split

row_id = pd.Index(range(20), name="row_id")
all_rows = pd.DataFrame({"target": [30 + i * 2 for i in row_id]}, index=row_id)
train_rows, test_rows = train_test_split(all_rows, test_size=0.20, random_state=42)
fit_rows, validation_rows = train_test_split(train_rows, test_size=0.25, random_state=43)

role = pd.Series("", index=all_rows.index, name="role")
role.loc[fit_rows.index] = "fit"
role.loc[validation_rows.index] = "validation"
role.loc[test_rows.index] = "test"
role_table = role.value_counts().rename_axis("role").to_frame("rows")

print("[구조로 확인] test를 먼저 보관하고 남은 train을 fit과 validation으로 나눕니다.")
display(role_table)
display(pd.DataFrame({"row_id": role.index, "role": role.values}).sort_values("row_id"))

colors = {"fit": "#2563eb", "validation": "#f59e0b", "test": "#dc2626"}
fig, ax = plt.subplots(figsize=(8, 2.7))
for name, group in role.groupby(role):
    ax.scatter(group.index, [1] * len(group), s=65, color=colors[name], label=name)
ax.set(xlabel="row id", yticks=[], title="Each row has exactly one role")
ax.legend(ncol=3)
plt.tight_layout()
plt.show()


## 전처리·ColumnTransformer·Pipeline과 평균 기준모형

숫자열과 범주열에는 서로 다른 변환이 필요합니다.

```text
숫자열 ─ 결측 대체 ─ 표준화 ─────┐
                                  ├─ 수치 입력행렬 ─ 예측모형
범주열 ─ 결측 대체 ─ one-hot ────┘
```

- `ColumnTransformer`: 열 집합별 전처리 경로를 지정합니다.
- `Pipeline`: 전처리와 예측모형의 `fit`·`predict` 순서를 결합합니다.
- 평균 기준모형: fit target의 평균 한 값을 모든 validation 행에 반복합니다.

대체값·표준화 기준·범주 목록은 fit에서만 계산하고 validation과 test에는 저장된
규칙만 적용합니다. 후보는 같은 validation 행과 같은 지표에서 평균 기준모형보다
오차를 줄였는지 비교합니다.


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

frame = pd.DataFrame({
    "power_factor": [80, np.nan, 88, 92, 84, 90, 86, np.nan],
    "load_type": ["Light", "Maximum", "Light", "Medium", "Maximum", "Medium", "Light", "Maximum"],
    "target": [42, 75, 48, 64, 71, 61, 50, 78],
})
X_fit = frame.loc[:5, ["power_factor", "load_type"]]
y_fit = frame.loc[:5, "target"]
X_validation = frame.loc[6:, ["power_factor", "load_type"]]
y_validation = frame.loc[6:, "target"]

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), ["power_factor"]),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), ["load_type"]),
])
candidate = Pipeline([("preprocess", clone(preprocess)), ("model", LinearRegression())])
baseline = DummyRegressor(strategy="mean")
candidate.fit(X_fit, y_fit)
baseline.fit(X_fit, y_fit)

candidate_prediction = candidate.predict(X_validation)
baseline_prediction = baseline.predict(X_validation)
comparison = pd.DataFrame({
    "actual": y_validation,
    "mean_baseline": baseline_prediction,
    "pipeline_prediction": candidate_prediction,
})
comparison["baseline_abs_error"] = (comparison["actual"] - comparison["mean_baseline"]).abs()
comparison["pipeline_abs_error"] = (comparison["actual"] - comparison["pipeline_prediction"]).abs()

print("[실행순서로 확인] Pipeline은 전처리와 예측모형을 한 fit·predict 경로로 결합합니다.")
print("fit에서 정한 숫자열 중앙값:", float(X_fit["power_factor"].median()))
print("fit target 평균인 baseline 상수:", float(y_fit.mean()))
print("Pipeline 단계:", list(candidate.named_steps))
display(comparison.round(2))
print("validation baseline MAE:", round(mean_absolute_error(y_validation, baseline_prediction), 2))
print("validation Pipeline MAE:", round(mean_absolute_error(y_validation, candidate_prediction), 2))


## 관련 연구와 이번 재현의 범위

Sathishkumar, Shin, and Cho (2021)는 같은 Steel 공개자료에서 여러 회귀모형의
에너지 사용 예측 성능을 비교했습니다.

- 논문: [*Efficient Energy Consumption Prediction Model for a Data Analytic-Enabled Industry Building in a Smart City*](https://doi.org/10.1080/09613218.2020.1809983)
- 재현: target·feature 정의, 전처리 Pipeline, 기준모형·선형회귀·Decision Tree의 동일 조건 비교
- 제외: 논문의 전체 전처리와 hyperparameter 탐색, 최고 성능 수치의 복제

같은 데이터셋을 사용한 것과 연구 전체를 재현한 것은 구분해 기록합니다.


## 예측 실험에서 남길 기록

1. 한 행의 의미, target, feature와 제외 열
2. test 보관과 train 내부 fit·validation 분할
3. 숫자열·범주열의 전처리와 규칙을 학습한 범위
4. 평균 기준모형과 후보모형의 동일 validation 비교
5. 선택 Pipeline 하나의 최종 test 결과
6. 무작위 holdout이 확인하지 못한 미래기간·외부시설 조건

모형 이름보다 자료의 역할과 비교조건을 먼저 고정해야 점수 차이를 해석할 수 있습니다.


## 실행 환경과 입력자료

| 재현 조건 | 확인할 출력 |
|---|---|
| 입력자료 | 실제 파일 경로와 파일명 |
| 자료 구조 | 데이터 소개에 제시한 예상 `shape`와 필수 열 |
| 실행환경 | Python과 주요 라이브러리 버전 |

다음 셀은 분석에 사용한 파일과 실행환경을 기록합니다. 파일 경로 또는 `shape`가 강의 기준과 다르면 입력 파일과 버전을 확인한 뒤 분석을 진행합니다.

> **실행 전 확인:** 이번 분석의 입력 파일과 예상 행 수는 무엇인가요?


In [ ]:
from pathlib import Path
import platform
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 130)
warnings.filterwarnings("ignore", category=FutureWarning)

def find_course_data(relative_path: str) -> Path:
    """0교시에서 준비한 data 폴더의 파일을 찾습니다."""
    relative = Path(relative_path)
    checked = []
    for start in [Path.cwd(), *Path.cwd().parents, Path("/content")]:
        candidate = (start / "data" / relative).resolve()
        if candidate in checked:
            continue
        checked.append(candidate)
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"데이터 파일을 찾지 못했습니다: {relative_path}. "
        "과정 폴더의 00_setup.ipynb를 먼저 실행하십시오."
    )
print("Python:", platform.python_version())
print("pandas:", pd.__version__)


## 선형회귀와 Decision Tree의 예측 규칙 비교

본 자료를 적합하기 전에 두 개의 작은 표로 알고리즘이 실제로 저장하는 규칙을 확인합니다.

1. 선형회귀: 네 관측에서 절편·기울기와 예측오차를 확인합니다.
2. Decision Tree: 온도 한 열을 한 번 나눈 뒤 각 leaf의 target 평균을 확인합니다.

실행 전에 선형회귀의 다음 예측값과 tree의 두 leaf 예측값을 먼저 계산해 보세요.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, export_text

linear_example = pd.DataFrame({
    "x": [1.0, 2.0, 3.0, 4.0],
    "actual_y": [10.0, 14.0, 18.0, 22.0],
})
tiny_linear = LinearRegression().fit(linear_example[["x"]], linear_example["actual_y"])
linear_example["predicted_y"] = tiny_linear.predict(linear_example[["x"]])
linear_example["error_y_minus_prediction"] = (
    linear_example["actual_y"] - linear_example["predicted_y"]
)
next_prediction = float(tiny_linear.predict(pd.DataFrame({"x": [5.0]}))[0])

print("[1] 선형회귀가 저장한 직선")
print(f"예측식: y_hat = {tiny_linear.intercept_:.1f} + {tiny_linear.coef_[0]:.1f} × x")
display(linear_example.round(2))
print(f"x=5의 예측값: {next_prediction:.1f}")

tree_example = pd.DataFrame({
    "temperature": [10, 12, 14, 20, 22, 24],
    "actual_usage": [40, 42, 44, 80, 82, 84],
})
tiny_tree = DecisionTreeRegressor(max_depth=1, min_samples_leaf=2, random_state=42)
tiny_tree.fit(tree_example[["temperature"]], tree_example["actual_usage"])
tree_example["leaf"] = tiny_tree.apply(tree_example[["temperature"]])
tree_example["tree_prediction"] = tiny_tree.predict(tree_example[["temperature"]])
leaf_summary = tree_example.groupby("leaf", as_index=False).agg(
    train_rows=("actual_usage", "size"),
    leaf_target_mean=("actual_usage", "mean"),
    tree_prediction=("tree_prediction", "first"),
)

print("\n[2] 깊이 1 Decision Tree가 저장한 조건")
print(export_text(tiny_tree, feature_names=["temperature"]))
display(tree_example)
print("각 leaf의 예측값은 그 leaf에 들어간 train target의 평균입니다.")
display(leaf_summary)

assert np.allclose(leaf_summary["leaf_target_mean"], leaf_summary["tree_prediction"])
print("작은 모형 원리 확인: PASS")


## 1. 변수 역할과 예측 시점

예측모형을 적합하기 전에 각 열의 역할을 분석 명세로 고정합니다.

| 역할 | Steel 실습의 열 | 판단 기준 |
|---|---|---|
| target $y$ | `Usage_kWh` | 예측할 결과인가 |
| numeric feature | 무효전력, power factor, `NSM` | 수치적 거리와 계산이 의미 있는가 |
| categorical feature | `WeekStatus`, `Day_of_week`, `Load_Type` | 범주 간 순서를 가정할 수 있는가 |
| 제외 | `CO2(tCO2)` | 결과 proxy이거나 예측 시점에 이용할 수 없는가 |

다음 출력은 역할표와 모델에 실제로 들어가는 열을 제시합니다. target이 X에 포함되지 않았는지, 숫자로 저장된 범주가 적절히 분류됐는지 확인합니다.

> **확인:** `Usage_kWh`와 산술적으로 연결된 결과 proxy를 X에 넣으면 평가가 왜 낙관적으로 변할까요?


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

DATA_PATH = find_course_data("steel/Steel_industry_data.csv")
data = pd.read_csv(DATA_PATH)
data["date"] = pd.to_datetime(data["date"], dayfirst=True)
data["hour"] = data["date"].dt.hour
analysis_name = "Steel energy regression"
target_name = "Usage_kWh"
numeric_features = ["Lagging_Current_Reactive.Power_kVarh", "Leading_Current_Reactive_Power_kVarh", "Lagging_Current_Power_Factor", "Leading_Current_Power_Factor", "NSM"]
categorical_features = ["WeekStatus", "Day_of_week", "Load_Type"]
feature_columns = numeric_features + categorical_features

print("shape:", data.shape)
print("target:", target_name)
print("excluded leakage-risk proxy: CO2(tCO2)")
display(data[[target_name, *feature_columns]].head(3))


### 출력 해석: 열의 역할 나누기

모든 열을 한꺼번에 넣지 않고 target·숫자 feature·범주 feature·제외 열을 구분합니다.


In [ ]:
feature_roles = pd.DataFrame({
    "role": ["target", "numeric features", "categorical features", "excluded leakage-risk proxy"],
    "columns": [target_name, ", ".join(numeric_features), ", ".join(categorical_features), "CO2(tCO2)"],
})
display(feature_roles)
print("모델에 실제로 들어가는 X 열:", feature_columns)
display(data[[target_name, *feature_columns]].head(5))


## 2. 전처리 규칙을 어디에서 배우고 어디에 적용하는가

이 Notebook에는 `train` 안에 후보 비교용 `validation`을 한 번 더 둡니다. 따라서 전처리 미리보기와 후보모형은 먼저 **fit 자료만 보고** 규칙을 배웁니다.

| 순서 | 사용하는 자료 | 실제로 하는 일 |
|---:|---|---|
| 1 | 전체 자료 | train 80%와 마지막 평가용 test 20% 분리 |
| 2 | train | fit과 validation으로 다시 분리 |
| 3 | fit | 수치열 중앙값·평균·표준편차, 범주 목록과 모형 규칙 학습 |
| 4 | validation | fit에서 배운 규칙을 그대로 적용해 세 후보의 MAE 비교 |
| 5 | 전체 train | 선택된 Pipeline을 다시 적합 |
| 6 | test | 선택된 Pipeline 하나의 최종 MAE를 한 번 계산 |

`ColumnTransformer`는 수치열과 범주열을 서로 다른 처리 경로로 보냅니다. `Pipeline`은 그 전처리와 모형을 한 묶음으로 만들어, 새 행을 예측할 때도 같은 순서와 같은 규칙이 적용되게 합니다.

다음 출력에서는 fit·validation·test의 행 수를 먼저 확인한 뒤, 같은 fit 행이 전처리 전의 혼합형 표에서 전처리 후의 숫자행렬로 어떻게 바뀌는지 비교합니다.

test 결과를 확인한 뒤 feature·모형·깊이를 바꾸면 그 test도 선택과정에 참여한 것입니다. 그런 변경을 평가하려면 새 외부 test 또는 이후 기간 자료를 별도로 사용해야 합니다.

> **확인:** validation이나 test에서 중앙값과 범주 목록을 새로 계산하면, 왜 평가자료가 학습과정에 들어간 셈이 될까요?


In [ ]:
X = data[feature_columns]
y = data[target_name]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
X_fit, X_validation, y_fit, y_validation = train_test_split(
    X_train, y_train, test_size=0.20, random_state=43
)

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), categorical_features),
])

print("fit/validation/test:", X_fit.shape, X_validation.shape, X_test.shape)
print("y fit/validation/test:", y_fit.shape, y_validation.shape, y_test.shape)


### 출력 해석: 같은 행이 전처리 전후에 어떻게 달라지는가

fit·validation·test가 서로 다른 행인지 확인합니다. 이어서 fit에서만 전처리 규칙을 학습하고, 같은 fit 행의 문자 범주와 숫자열이 실제 모델 입력표에서 어떻게 바뀌는지 나란히 봅니다.


In [ ]:
from sklearn.base import clone

split_preview = pd.concat([
    X_fit.head(3).assign(target=y_fit.head(3), split="fit"),
    X_validation.head(3).assign(target=y_validation.head(3), split="validation"),
    X_test.head(3).assign(target=y_test.head(3), split="test"),
])
print("[1] fit·validation·test는 같은 feature 열을 사용하지만 서로 다른 행입니다.")
display(split_preview[["split", "target", *feature_columns[:4]]])

preview_preprocess = clone(preprocess)
preview_matrix = preview_preprocess.fit_transform(X_fit)
preview_columns = preview_preprocess.get_feature_names_out()
transformed_fit = pd.DataFrame(
    preview_matrix,
    index=X_fit.index,
    columns=preview_columns,
)

preview_numeric = numeric_features[0]
preview_category = categorical_features[-1]
encoded_category_columns = [
    column for column in preview_columns
    if column.startswith(f"cat__{preview_category}_")
]
transformed_columns = [f"num__{preview_numeric}", *encoded_category_columns]

print("\n[2] 전처리 전: 숫자값과 문자 범주가 있는 원자료")
display(X_fit.loc[X_fit.head(5).index, [preview_numeric, preview_category]])
print("[3] 전처리 후: 같은 index가 표준화 숫자와 범주별 0/1 열로 바뀐 입력표")
display(transformed_fit.loc[X_fit.head(5).index, transformed_columns].round(3))

numeric_medians = pd.DataFrame({
    "numeric feature": numeric_features,
    "fit에서 배운 중앙값": preview_preprocess
        .named_transformers_["num"]
        .named_steps["imputer"]
        .statistics_,
})
print("[4] 숫자열에 빈 값이 생기면 사용할 값도 validation이나 test가 아니라 fit에서 배웁니다.")
display(numeric_medians.round(3))

pipeline_roles = pd.DataFrame({
    "도구": ["ColumnTransformer", "Pipeline"],
    "역할": [
        "숫자열과 범주열을 서로 다른 전처리 경로로 보냄",
        "전처리 → 예측모형의 실행 순서를 하나로 묶음",
    ],
    "predict 때의 동작": [
        "fit에서 배운 변환만 validation과 test에 적용",
        "validation·test 변환 뒤 모델 예측까지 같은 순서로 실행",
    ],
})
display(pipeline_roles)


## 3. 평균 기준모형과 후보모형 비교

평균 기준모형은 fit target의 평균 한 값을 모든 validation 행에 반복합니다. 후보모형의 점수는 이 기준과 같은 행·지표에서 비교합니다.

| 출력 | 확인할 근거 |
|---|---|
| train target 평균 | 기준예측의 출처 |
| 기준모형 상수 | train 평균과의 일치 여부 |
| 첫 6개 validation 행 | 실제값·세 예측값·절대오차 |
| 후보 선택표 | 평균 기준모형·선형회귀·Decision Tree의 validation MAE |

MAE는 Steel 자료에서 kWh 단위입니다. 후보 MAE가 더 작다는 사실은 이 validation에서 입력변수의 예측정보가 평균 한 값보다 유용했다는 근거입니다.

> **확인:** 입력변수가 다른 여섯 행에도 평균 기준모형의 예측이 같은 이유는 무엇일까요?


In [ ]:
models = {
    "mean baseline": DummyRegressor(strategy="mean"),
    "linear regression": Pipeline([
        ("preprocess", clone(preprocess)),
        ("model", LinearRegression()),
    ]),
    "decision tree": Pipeline([
        ("preprocess", clone(preprocess)),
        ("model", DecisionTreeRegressor(
            max_depth=8,
            min_samples_leaf=10,
            random_state=42,
        )),
    ]),
}

# 후보 비교에는 원래 test가 아니라 train 내부 validation만 사용합니다.
selection_rows = []
fitted_candidates = {}
validation_predictions = {}
for name, model in models.items():
    candidate = clone(model)
    candidate.fit(X_fit, y_fit)
    prediction = candidate.predict(X_validation)
    selection_rows.append({
        "model": name,
        "validation_mae": mean_absolute_error(y_validation, prediction),
    })
    fitted_candidates[name] = candidate
    validation_predictions[name] = prediction

validation_table = (
    pd.DataFrame(selection_rows)
    .sort_values("validation_mae")
    .reset_index(drop=True)
)
selected_model_name = validation_table.loc[0, "model"]

# Validation 진단과 Practice 확장에서 같은 예측 배열을 재사용합니다.
baseline_model = fitted_candidates["mean baseline"]
baseline_value = float(np.asarray(baseline_model.constant_).reshape(-1)[0])
baseline_prediction = validation_predictions["mean baseline"]
linear_prediction = validation_predictions["linear regression"]
tree_prediction = validation_predictions["decision tree"]

print("[train 내부 validation에서 세 후보 비교]")
display(validation_table.round(3))
print("validation MAE로 정한 최종 평가 후보:", selected_model_name)

# 선택을 끝낸 뒤 전체 train에 다시 적합하고 test를 한 번만 평가합니다.
final_model = clone(models[selected_model_name])
final_model.fit(X_train, y_train)
final_test_prediction = final_model.predict(X_test)
final_test_table = pd.DataFrame([{
    "selected_model": selected_model_name,
    "test_mae": mean_absolute_error(y_test, final_test_prediction),
    "test_rows": len(y_test),
}])
print("test 결과를 본 뒤 feature·모형·깊이를 바꾸려면 새 외부 test 또는 이후 기간 자료가 필요합니다.")

# 뒤의 검토 프롬프트가 사용할 호환 이름입니다. 이 표에는 test 점수가 없습니다.
result_table = validation_table.copy()


### 출력 해석: 평균 기준예측의 숫자부터 후보 모델까지

평균 baseline이 fit target의 평균 하나를 모든 validation 행에 반복하는지 먼저 확인합니다. 그다음 같은 validation 행에서 각 후보의 예측과 절대오차를 나란히 비교합니다.


In [ ]:
baseline_model = fitted_candidates["mean baseline"]
baseline_value = float(np.asarray(baseline_model.constant_).reshape(-1)[0])
baseline_prediction = validation_predictions["mean baseline"]
linear_prediction = validation_predictions["linear regression"]
tree_prediction = validation_predictions["decision tree"]

print(f"fit target 평균                         : {y_fit.mean():.3f}")
print(f"baseline이 validation에 반복하는 예측값: {baseline_value:.3f}")

prediction_view = pd.DataFrame({
    "actual": y_validation.to_numpy()[:6],
    "mean_baseline": baseline_prediction[:6],
    "baseline_abs_error": np.abs(
        y_validation.to_numpy()[:6] - baseline_prediction[:6]
    ),
    "linear_prediction": linear_prediction[:6],
    "tree_prediction": tree_prediction[:6],
}, index=y_validation.index[:6])

print("[같은 validation 행에서 실제값과 세 후보 예측 비교]")
display(prediction_view.round(3))
print("[후보 선택에 사용한 validation MAE]")
display(validation_table.round(3))
print("[선택을 끝낸 뒤 한 번 계산한 최종 test MAE]")
display(final_test_table.round(3))


## 4. 예측값·오차·실험조건 검증

평가지표는 여러 행의 오류를 요약합니다. 선택 후보의 실제값과 예측값을 같은 index에 연결해 요약값의 근거를 확인합니다.

| 검증 항목 | 통과 조건 |
|---|---|
| target 제외 | `Usage_kWh`가 X의 열에 없음 |
| 행 정렬 | 예측 배열과 `y_test`의 길이·순서가 일치 |
| 비교 대상 | 세 후보가 같은 validation index 사용 |
| 지표 | 모든 MAE가 유한한 kWh 값 |
| 기준모형 | 세 후보의 MAE는 동일 validation에서 비교하고 test에는 선택 후보 하나만 적용 |

검증 통과는 코드와 비교조건의 일관성을 확인합니다. 미래기간 성능과 다른 공장의 일반화는 별도 평가 대상입니다.


In [ ]:
print("최종 test에는 validation에서 정한 Pipeline 하나만 적용했습니다.")
examples = pd.DataFrame({
    "observed": y_test.to_numpy()[:8],
    "predicted": final_test_prediction[:8],
}, index=y_test.index[:8])
display(examples.round(3))

checks = pd.Series({
    "target excluded from X": target_name not in X.columns,
    "fit and validation disjoint": X_fit.index.intersection(X_validation.index).empty,
    "train and test disjoint": X_train.index.intersection(X_test.index).empty,
    "three validation candidates": len(validation_table) == 3,
    "finite validation MAE": np.isfinite(validation_table["validation_mae"]).all(),
    "one selected model evaluated on test": len(final_test_table) == 1,
    "test prediction aligned": len(final_test_prediction) == len(y_test),
    "finite final test MAE": np.isfinite(final_test_table["test_mae"]).all(),
}, name="passed")
display(checks.to_frame())
assert checks.all()
print("예측 Pipeline 검증: PASS")


## ChatGPT 실시간 검토: 예측실험의 설계 오류 찾기

이번에는 새로운 모델 코드를 요청하기 전에 **예측대상, 입력변수, 제외열, 자료 분할, 전처리와 기준모형**이 일관적인지 점검합니다.

ChatGPT가 제안한 열이 실제로 존재하는지, 결과에서 파생된 누수 열은 아닌지, 평가자료의 정보가 전처리 학습에 들어가지 않는지를 Notebook 출력으로 확인합니다.


In [ ]:
excluded_columns = [
    column for column in ["CO2(tCO2)", "casual", "registered"]
    if column in data.columns
]
split_audit = pd.DataFrame({
    "split": ["fit", "validation", "test"],
    "rows": [len(X_fit), len(X_validation), len(X_test)],
})
preprocess_audit = pd.DataFrame({
    "column_type": ["numeric", "categorical"],
    "columns": [", ".join(numeric_features), ", ".join(categorical_features)],
    "fit에서 학습하는 규칙": [
        "median imputation and standardization",
        "most-frequent imputation and one-hot category list",
    ],
})

review_prompt = f"""
다음 회귀 예측실험의 설계를 감사해 주세요.

- 분석: {analysis_name}
- 예측대상: {target_name}
- 입력변수: {feature_columns}
- 결과누수 위험 때문에 제외한 열: {excluded_columns}
- 자료 분할(행 수):
{split_audit.to_string(index=False)}
- 분할 seed: 전체→train/test 42, train→fit/validation 43
- 전처리 명세:
{preprocess_audit.to_string(index=False)}
- validation 후보 비교표:
{validation_table.to_string(index=False)}
- 선택된 Pipeline의 최종 test 보고표:
{final_test_table.to_string(index=False)}

아래 표 형식으로 답해 주세요: 점검항목 | 현재 판단 | 확인 근거 | 다음 행동
점검항목은 예측시점, 제외열과 결과누수, 세 자료의 역할, 전처리 적합 범위,
평균 기준모형, validation MAE 비교, 최종 test 행 수를 포함해 주세요.

제약: 새 모델을 추천하거나 코드를 길게 작성하지 말고, 제공된 열·행 수·표에
없는 수치를 만들지 마세요. 가장 먼저 수정할 설계 위험 한 가지만 마지막에 제시해 주세요.
이미 본 test 결과를 근거로 feature·모형·깊이를 바꾸지 마세요. 변경이 필요하다면
새 외부 test 또는 이후 기간 자료가 필요하다는 점을 답변에 포함해 주세요.
""".strip()
print("[ChatGPT에 복사할 프롬프트]\n")
print(review_prompt)


### 실시간 ChatGPT 답변을 받은 뒤

1. 제안된 입력열을 실제 스키마와 제외열 목록에 대조합니다.
2. 결측 대체·표준화·범주 변환이 학습자료에서만 적합되는지 확인합니다.
3. 변경 코드는 기존 결과를 덮지 않는 새 셀에서 한 가지씩 시험합니다.


## 해석 범위

이번 결과는 같은 기간의 관측을 무작위로 나눈 holdout에서 전처리와 예측 Pipeline의 작동, 그리고 기준모형 대비 오차를 평가합니다.

시간순 미래예측, 다른 공장·도시로의 일반화와 인과효과를 검증한 결과는 아닙니다. 해당 질문은 시간 경계, 외부표본과 별도의 연구설계를 사용해 평가해야 합니다.


## 오늘의 결론과 다음 단계

### 이번 교시의 결론

예측대상과 입력변수를 정하고 자료의 역할을 나눈 뒤, train에서 배운 전처리와 모형을 Pipeline으로 묶어 기준모형과 비교합니다.

### 적용 순서

1. `05_sklearn_pipeline_follow.ipynb`에서 핵심 시연을 같은 조건으로 재현합니다.
2. `05_sklearn_pipeline_practice.ipynb`에서 지정된 데이터 또는 조건 하나만 바꿔 비교합니다.
3. 변경조건, 비교표 또는 그림, 결과문장과 현재 검증하지 못한 범위를 함께 기록합니다.

### 실습 중 확인할 질문

- 이 지도학습 회귀 문제에서 feature `X`, target `y`, 예측값 `ŷ`과 오차 `y-ŷ`는 각각 무엇입니까?
- 선형회귀는 어떤 기준으로 계수를 정하고, Decision Tree는 어떤 기준으로 분할점과 leaf 예측값을 정합니까?
- 표준화·결측 대체를 train에서만 학습해야 하는 이유와, 평균 baseline·후보모형을 공정하게 비교할 때 고정할 조건은 무엇입니까?

마지막 `PASS`는 Notebook에 적힌 실행조건을 확인한 결과이며 연구결론의 타당성을 대신하지 않습니다.
